In [37]:
# =============================================================================
# IMPORTS
# =============================================================================
from pathlib import Path
from prophet import Prophet

import warnings
import prophet
from prophet.diagnostics import cross_validation, performance_metrics
import matplotlib as plt

import pandas as pd
import numpy as np
import seaborn as sns
from loguru import logger
import sys

warnings.filterwarnings("ignore", category=FutureWarning)
logger.remove()
logger.add(sys.stderr, level="INFO", format="{time:HH:mm:ss} | {level:<7} | {message}")
logger.info("FE Avance 2 — EpiForecast-MX inicializado")

12:50:04 | INFO    | FE Avance 2 — EpiForecast-MX inicializado


In [2]:
# =============================================================================
# CONSTANTES Y CONFIGURACIÓN
# =============================================================================

# --- Rutas -------------------------------------------------------------------
# Dataset ya preparado para hacer merge con INEGI (sale del pipeline de limpieza/transform)
DATA_PATH = Path("../data/processed/data_inegi_General.csv")

# --- Paleta IMSS institucional (Guía cromática oficial) ----------------------
IMSS_COLORS = {
    "neutral_black":  "#231F20",  # PANTONE Neutral Black C
    "burgundy":       "#9B2242",  # PANTONE 7420 C
    "dark_burgundy":  "#6F1D46",  # PANTONE 7421 C
    "cool_gray":      "#97999B",  # PANTONE Cool Gray C
    "teal":           "#00524E",  # PANTONE IMSS 561 C
    "dark_teal":      "#173F35",  # PANTONE 627 C
    "cream":          "#E8D5B5",  # PANTONE 7402 C
    "gold":           "#B58500",  # PANTONE 1255 C
}

# Paleta secuencial para gráficos
PALETTE_MAIN = [
    IMSS_COLORS["teal"],
    IMSS_COLORS["burgundy"],
    IMSS_COLORS["gold"],
    IMSS_COLORS["dark_teal"],
    IMSS_COLORS["dark_burgundy"],
    IMSS_COLORS["cool_gray"],
    IMSS_COLORS["neutral_black"],
    IMSS_COLORS["cream"],
]

PALETTE_PADECIMIENTO = {
    "Depresión":  IMSS_COLORS["burgundy"],
    "Parkinson":  IMSS_COLORS["teal"],
    "Alzheimer":  IMSS_COLORS["gold"],
}

PALETTE_SEXO = {
    "Hombres": IMSS_COLORS["teal"],
    "Mujeres": IMSS_COLORS["burgundy"],
}

# --- Estilo global de matplotlib ---------------------------------------------
plt.rcParams.update({
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.edgecolor":      IMSS_COLORS["cool_gray"],
    "axes.labelcolor":     IMSS_COLORS["neutral_black"],
    "text.color":          IMSS_COLORS["neutral_black"],
    "xtick.color":         IMSS_COLORS["neutral_black"],
    "ytick.color":         IMSS_COLORS["neutral_black"],
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "grid.color":          IMSS_COLORS["cool_gray"],
    "font.family":         "sans-serif",
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "figure.titlesize":    15,
    "figure.titleweight":  "bold",
    "figure.dpi":          120,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
})

logger.success(f"Configuración cargada | Paleta IMSS: {len(IMSS_COLORS)} colores")

12:30:25 | SUCCESS | Configuración cargada | Paleta IMSS: 8 colores


In [3]:
df_datos = pd.read_csv(DATA_PATH)
df_datos['Fecha'] = pd.to_datetime(df_datos['Fecha'])
df_datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60384 entries, 0 to 60383
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Padecimiento                        60384 non-null  object        
 1   Semana                              60384 non-null  int64         
 2   Fecha                               60384 non-null  datetime64[ns]
 3   Entidad                             60384 non-null  object        
 4   incrementos_hombres                 60384 non-null  int64         
 5   incrementos_mujeres                 60384 non-null  int64         
 6   Region                              60384 non-null  object        
 7   Superficie_km2                      60384 non-null  float64       
 8   Hombres                             60384 non-null  int64         
 9   Mujeres                             60384 non-null  int64         
 10  Total                 

In [30]:
serie  = (df_datos
                .groupby(["Fecha","region_salud_mental"])[["incrementos_hombres", "incrementos_mujeres"]]
                .sum()
                .reset_index()
                .rename(columns={'Fecha':'ds'})
            )

serie["y"] = serie["incrementos_hombres"] + serie["incrementos_mujeres"]
serie = serie.sort_values('ds')
regiones = serie['region_salud_mental'].unique()
serie.head(5)

,ds,region_salud_mental,incrementos_hombres,incrementos_mujeres,y
0,2013-12-30,Metropolitana alta,2,11,13
1,2013-12-30,Rural / dispersa,12,17,29
2,2013-12-30,Sur-Sureste vulnerable,9,23,32
3,2013-12-30,Urbana media,13,59,72
4,2014-01-06,Metropolitana alta,51,186,237


In [51]:
serie_prophet = serie.loc[serie['region_salud_mental']=='Metropolitana alta',['ds','y']].copy()

In [53]:

resultados = []

for region in regiones:
    modelo = Prophet()
    serie_prophet = serie.loc[serie['region_salud_mental']==region,['ds','y']].copy()
    modelo.fit(serie_prophet)

    cv_prophet = cross_validation(
        modelo,
        initial='360 days',    
        period='30 days',
        horizon='90 days'
    )

    df_pm = performance_metrics(cv_prophet)
    resumen = df_pm[['rmse', 'mae', 'mape', 'mdape']].mean()
    fila = [region] + resumen.tolist()
    resultados.append(fila)


13:09:25 - cmdstanpy - INFO - Chain [1] start processing
13:09:25 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
  0%|          | 0/132 [00:00<?, ?it/s]13:09:25 - cmdstanpy - INFO - Chain [1] start processing
13:09:25 - cmdstanpy - INFO - Chain [1] done processing
  1%|          | 1/132 [00:00<00:31,  4.10it/s]13:09:25 - cmdstanpy - INFO - Chain [1] start processing
13:09:25 - cmdstanpy - INFO - Chain [1] done processing
  2%|▏         | 2/132 [00:00<00:35,  3.62it/s]13:09:25 - cmdstanpy - INFO - Chain [1] start processing
13:09:25 - cmdstanpy - INFO - Chain [1] done processing
  2%|▏         | 3/132 [00:00<00:32,  3.95it/s]13:09:25 - cmdstanpy - INFO - Chain [1] start processing
13:09:26 - cmdstanpy - INFO - Chain [1] done processing
  3%|▎         | 4/132 [00:01<00:32,  3.94it/s]13:09:26 - cmdstanpy - INFO - Chain [1] start processing
13:09:26 - cmdstanpy - INFO - Chain [1] done pr

In [55]:
df_resultados = pd.DataFrame(resultados, columns=['region', 'rmse', 'mae', 'mape', 'mdape'])
print(df_resultados)


                   region        rmse         mae      mape     mdape
0      Metropolitana alta   94.165812   61.454872  0.131196  0.066719
1        Rural / dispersa  109.236189   79.039089  0.217800  0.132385
2  Sur-Sureste vulnerable   70.993406   47.279399  0.261685  0.154316
3            Urbana media  196.654183  134.392048  0.178378  0.100823
